In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field

In [4]:
llm=ChatOpenAI(
    model="openai/gpt-4o-mini",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    max_completion_tokens=200
)

In [10]:
message=[
    SystemMessage("You are a terse assistant who answers in exactly five words."),
    HumanMessage("What is the capital of France?"),
]

llm.invoke(message).content



'Paris is the capital of France.'

In [22]:
@tool
def get_share_price(symbol: str) -> float:
    """Return the current share price for a given ticker symbol."""
    fake_prices = {"AAPL": 241.5, "GOOG": 168.2, "AMZN": 198.0}
    return fake_prices.get(symbol.upper(), 0.0)

In [40]:
@tool
def city_prices(city:str)-> float:
    """Return the current city price for the given city prices """
    fake_city_prices={"KOLKATA":55.5,"MUMBAI":44.2}
    return fake_city_prices.get(city.upper(),0.0)


In [41]:
llm_with_tools=llm.bind_tools([get_share_price,city_prices])

In [42]:
# Start the conversation and keep the model's tool request in the history
conversation = [HumanMessage("What is the prices of kolkata city?")]
ai_message = llm_with_tools.invoke(conversation)
print(ai_message)

conversation.append(ai_message)


# Run each requested tool and add its result as a ToolMessage
for call in ai_message.tool_calls:
    if call["name"] == "get_share_price":
        result = get_share_price.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
    elif call["name"]=="city_prices":
        result=city_prices.invoke(call["args"])
        conversation.append(ToolMessage(content=str(result), tool_call_id=call["id"]))
# Invoke again, now that the model can see the tool result
final = llm_with_tools.invoke(conversation)
print(final.content)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 83, 'total_tokens': 99, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': 0, 'cached_tokens': 0, 'video_tokens': 0}, 'cost': 2.205e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 2.205e-05, 'upstream_inference_prompt_cost': 1.245e-05, 'upstream_inference_completions_cost': 9.6e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-4o-mini', 'system_fingerprint': 'fp_b99562ea4c', 'id': 'gen-1790438914-hHcOj06R4MpZqQuOp86o', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--01a0de79-b533-7652-8dc9-08626af45826-0' tool_calls=[{'name': 'city_prices', 'args': {'city': 'Kolkata'}, 'id': 'call_gIybqqSB6VjdtOZQvDdLPrbR', 'type': 'to

In [43]:
class Company(BaseModel):
    name: str = Field(description="The company name")
    ticker: str = Field(description="The stock ticker symbol")
    founded_year: int = Field(description="The year the company was founded")

structured_llm = llm_with_tools.with_structured_output(Company)

company = structured_llm.invoke("Tell me about Amazon the technology company")
print(company)
print("Just the ticker:", company.ticker)

name='Amazon.com, Inc.' ticker='AMZN' founded_year=1994
Just the ticker: AMZN
